# TMDB Marvel movie-review collection

This clean notebook collects reviews, release dates, and review timestamps for the selected Marvel films. Run the cells from top to bottom. The collection cell prints progress and may take a few minutes when it collects every review page.

In [20]:
# Install the project dependencies once with: python -m pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from getpass import getpass
from pathlib import Path
import os
import time

import pandas as pd
import requests

# The prompt keeps the API key out of the notebook file.
API_KEY = os.environ.get('TMDB_API_KEY') or getpass('Enter your TMDB API key: ')
if not API_KEY:
    raise ValueError('A TMDB API key is required.')

BASE_URL = 'https://api.themoviedb.org/3'
session = requests.Session()

# title, release year, franchise
MOVIES = [
    ('Spider-Man', 2002, 'Spider-Man'),
    ('Spider-Man 2', 2004, 'Spider-Man'),
    ('Spider-Man 3', 2007, 'Spider-Man'),
    ('The Amazing Spider-Man', 2012, 'Spider-Man'),
    ('The Amazing Spider-Man 2', 2014, 'Spider-Man'),
    ('Spider-Man: Homecoming', 2017, 'Spider-Man'),
    ('Spider-Man: Far From Home', 2019, 'Spider-Man'),
    ('Spider-Man: No Way Home', 2021, 'Spider-Man'),
    ('Spider-Man: Brand New Day', 2026, 'Spider-Man'),
    ('Spider-Man: Into the Spider-Verse', 2018, 'Spider-Man'),
    ('Spider-Man: Across the Spider-Verse', 2023, 'Spider-Man'),
    ('The Avengers', 2012, 'Avengers'),
    ('Avengers: Age of Ultron', 2015, 'Avengers'),
    ('Avengers: Infinity War', 2018, 'Avengers'),
    ('Avengers: Endgame', 2019, 'Avengers'),
    ('Captain America: The First Avenger', 2011, 'Captain America'),
    ('Captain America: The Winter Soldier', 2014, 'Captain America'),
    ('Captain America: Civil War', 2016, 'Captain America'),
    ('Iron Man', 2008, 'Iron Man'),
    ('Iron Man 2', 2010, 'Iron Man'),
    ('Iron Man 3', 2013, 'Iron Man'),
]

MAX_PAGES_PER_MOVIE = None

In [22]:
def tmdb_get(endpoint, **params):
    response = session.get(
        f'{BASE_URL}{endpoint}',
        params={'api_key': API_KEY, **params},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def find_movie(title, year):
    search = tmdb_get('/search/movie', query=title, year=year)
    candidates = search.get('results', [])
    if not candidates:
        raise LookupError(f'No TMDB result found for {title} ({year}).')
    selected = next(
        (item for item in candidates if str(item.get('release_date', '')).startswith(str(year))),
        candidates[0],
    )
    return tmdb_get(f"/movie/{selected['id']}")


def get_reviews(movie_id, max_pages=None):
    reviews = []
    page = 1
    while max_pages is None or page <= max_pages:
        data = tmdb_get(f'/movie/{movie_id}/reviews', language='en-US', page=page)
        reviews.extend(data.get('results', []))
        total_pages = data.get('total_pages', 0)
        print(f'    review page {page}/{total_pages}; {len(reviews)} collected')
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.25)
    return reviews

In [23]:
all_reviews = []

for index, (title, year, franchise) in enumerate(MOVIES, start=1):
    print(f'[{index}/{len(MOVIES)}] Finding {title} ({year})')
    movie = find_movie(title, year)
    print(f"  Matched: {movie['title']} | release date: {movie.get('release_date')}")
    movie_reviews = get_reviews(movie['id'], MAX_PAGES_PER_MOVIE)

    for review in movie_reviews:
        all_reviews.append({
            'movie_id': movie['id'],
            'movie': movie['title'],
            'release_date': movie.get('release_date'),
            'franchise': franchise,
            'review_id': review.get('id'),
            'review': review.get('content'),
            'review_date': review.get('created_at'),
        })

reviews_df = pd.DataFrame(all_reviews)
if reviews_df.empty:
    raise RuntimeError('TMDB returned no reviews; no CSV was created.')

reviews_df['release_date'] = pd.to_datetime(reviews_df['release_date'], errors='coerce')
reviews_df['review_date'] = pd.to_datetime(reviews_df['review_date'], errors='coerce', utc=True)
reviews_df['review_date'] = reviews_df['review_date'].dt.tz_localize(None).dt.normalize()
reviews_df['days_since_release'] = (
    reviews_df['review_date'] - reviews_df['release_date']
).dt.days

print(f'Collected {len(reviews_df):,} reviews.')
print(f'Missing release dates: {reviews_df["release_date"].isna().sum():,}')
print(f'Reviews before release: {(reviews_df["days_since_release"] < 0).sum():,}')
display(reviews_df.head())

[1/21] Finding Spider-Man (2002)
  Matched: Spider-Man | release date: 2002-05-01
    review page 1/2; 20 collected
    review page 2/2; 34 collected
[2/21] Finding Spider-Man 2 (2004)
  Matched: Spider-Man 2 | release date: 2004-06-25
    review page 1/2; 20 collected
    review page 2/2; 32 collected
[3/21] Finding Spider-Man 3 (2007)
  Matched: Spider-Man 3 | release date: 2007-05-01
    review page 1/2; 20 collected
    review page 2/2; 34 collected
[4/21] Finding The Amazing Spider-Man (2012)
  Matched: The Amazing Spider-Man | release date: 2012-06-23
    review page 1/1; 4 collected
[5/21] Finding The Amazing Spider-Man 2 (2014)
  Matched: The Amazing Spider-Man 2 | release date: 2014-04-16
    review page 1/1; 6 collected
[6/21] Finding Spider-Man: Homecoming (2017)
  Matched: Spider-Man: Homecoming | release date: 2017-07-05
    review page 1/1; 11 collected
[7/21] Finding Spider-Man: Far From Home (2019)
  Matched: Spider-Man: Far From Home | release date: 2019-06-28
    revi

,movie_id,movie,release_date,franchise,review_id,review,review_date,days_since_release
0,557,Spider-Man,2002-05-01,Spider-Man,5c8429f592514127691f8310,Sam Raimi's Spider-Man captures the spirit of ...,2019-03-09,6156
1,557,Spider-Man,2002-05-01,Spider-Man,5dc3308b470ead00158c8738,So many Spiderman movies out there but this wi...,2019-11-06,6398
2,557,Spider-Man,2002-05-01,Spider-Man,5dc38a688d22fc00183d1510,This is one of the few films that you can watc...,2019-11-07,6399
3,557,Spider-Man,2002-05-01,Spider-Man,5dc393ac9d89390015350524,I keep telling people that this is the real Sp...,2019-11-07,6399
4,557,Spider-Man,2002-05-01,Spider-Man,5dc3999b7d2bc100173d1175,Films from the 2000s really are way different ...,2019-11-07,6399


In [ ]:
project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'DATA').is_dir():
    project_root = project_root.parent
data_directory = project_root / 'DATA'
data_directory.mkdir(exist_ok=True)
output_path = data_directory / 'marvel_movie_reviews.csv'
reviews_df.to_csv(output_path, index=False)
print(f'Saved {len(reviews_df):,} reviews to {output_path}')

# This download happens only in Google Colab.
try:
    from google.colab import files
    files.download(str(output_path))
except ModuleNotFoundError:
    pass

Saved 486 reviews to C:\Users\andre\OneDrive\Documents\CS Personal Projects\DS-4002-Project-1\DATA\marvel_movie_reviews.csv


: 